# Training ViTVS untuk Bird Sound Denoising (Sesuai Paper ViTVS)

Notebook ini mengikuti paper *"Vision Transformer Segmentation for Visual Bird Sound Denoising" (ViTVS)*.
Model ViTVS menggunakan arsitektur Vision Transformer (Encoder 12 layer, Decoder 12 layer) untuk melakukan segmentasi pada gambar spectrogram.

**Spesifikasi Training (Berdasarkan Paper 4.1. Implementation details):**
- Input: 256 × 256 × 3
- Batch Size: 8
- Epochs: 100
- Optimizer: AdamW (lr=5e-5, weight_decay=5e-4)
- Loss Function: Negative Log-Likelihood (NLL) Loss

Pastikan Runtime Google Colab diatur ke **GPU**.

## 1. Setup & Download Dataset
Dataset yang digunakan berasal dari dataset BirdSoundsDenoising.



In [ ]:
# Clone repositori Anda dan install dependency
!git clone https://github.com/user312982/bird-denoising-sound.git
%cd bird-denoising-sound
!pip install torchaudio torchvision matplotlib Pillow

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

# Import custom modules
from models.dataset import BirdSoundsSegmentationDataset
from models.vitvs import ViTVS_Segmenter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {device}")

In [ ]:
# Download Dataset (Train.zip dan Valid.zip)
!wget -q -O Train.zip "https://zenodo.org/records/7191406/files/Train.zip?download=1"
!wget -q -O Valid.zip "https://zenodo.org/records/7191406/files/Valid.zip?download=1"

# Ekstrak
!unzip -q Train.zip -d dataset/
!unzip -q Valid.zip -d dataset/
!rm Train.zip Valid.zip

print("Dataset siap!")

## 2. Dataloader & Visualisasi Dataset

Dalam ViTVS, dataset berupa gambar *Spectrogram* dan label *Binary Mask*. Kita resize ke 256x256 sesuai paper.

In [ ]:
train_img_dir = 'dataset/Train/Images'
train_mask_dir = 'dataset/Train/Masks'
val_img_dir = 'dataset/Valid/Images'
val_mask_dir = 'dataset/Valid/Masks'

# Resolusi input sesuai paper ViTVS (256x256)
img_size = (256, 256)

# Jika Colab RAM crash, ganti num_workers=0 dan batch_size=4
batch_size = 8
num_workers = 2

train_dataset = BirdSoundsSegmentationDataset(
    images_dir=train_img_dir,
    masks_dir=train_mask_dir,
    img_size=img_size,
    augment=True
)

val_dataset = BirdSoundsSegmentationDataset(
    images_dir=val_img_dir,
    masks_dir=val_mask_dir,
    img_size=img_size,
    augment=False
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(f"Total Sampel Train: {len(train_dataset)}")
print(f"Total Sampel Validasi: {len(val_dataset)}")

## 3. Inisialisasi Model, Loss, dan Optimizer

In [ ]:
# 1. Model ViTVS (Encoder 12 layer, Decoder 12 layer)
# Menggunakan patch_size 16 sesuai standar ViT
model = ViTVS_Segmenter(
    img_size=img_size, 
    patch_size=16, 
    in_chans=3, 
    embed_dim=768, 
    depth=12, 
    num_heads=12, 
    num_classes=2
).to(device)

# 2. Loss Function (NLL Loss seperti di paper Eq. 18)
# NLLLoss membutuhkan input berupa log-probabilities
criterion = nn.NLLLoss().to(device)

# 3. Optimizer (AdamW dengan lr=5e-5 dan weight_decay=5e-4 sesuai paper)
optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=5e-4)

# Sesuai training iteration di paper
epochs = 100

## 4. Training Loop Model ViTVS

In [ ]:
best_val_loss = float('inf')
save_path = "best_vitvs_segmenter.pth"

print("Mulai Training...")
for epoch in range(epochs):
    # --- TRAINING ---
    model.train()
    train_loss = 0.0
    
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
        images = images.to(device)
        masks = masks.to(device) # target harus bernilai 0 atau 1 (long tensor)
        
        optimizer.zero_grad()
        
        # Forward pass
        logits = model(images)
        
        # Log-Softmax untuk NLL Loss
        log_probs = torch.nn.functional.log_softmax(logits, dim=1)
        
        # Calculate NLL loss
        loss = criterion(log_probs, masks)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        
    train_loss = train_loss / len(train_dataset)
    
    # --- VALIDATION ---
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid]"):
            images = images.to(device)
            masks = masks.to(device)
            
            logits = model(images)
            log_probs = torch.nn.functional.log_softmax(logits, dim=1)
            loss = criterion(log_probs, masks)
            val_loss += loss.item() * images.size(0)
            
    val_loss = val_loss / len(val_dataset)
    
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), save_path)
        print(f"  [+] Model ViTVS terbaik disimpan ke '{save_path}'")

print("Training Selesai!")

## 5. Denoising / Inference Process (Tahap ke-2)

Setelah model segmentasi dilatih, kita menggunakannya untuk men-denoise audio baru.
Gunakan script `inference_vitvs.py` secara lokal atau di cell berikutnya.